# FINAL · mã hóa khóa hiếm

Thử thêm **2 cột** vào nhánh `XGB-D03` và đối chiếu tác động lên tổ hợp `FINAL`.

## Mô hình đối chứng

`FINAL` gốc ghép `GLM-NEWTON` với `XGB-D03` bằng logistic ở tầng gộp. Dữ liệu chính thức có 699.960 dòng train. Nhánh XGBoost dùng cùng 5 fold và huấn luyện trên RTX 4060.

## Tạo đặc trưng

### RARE-TE · 2 cột số tiền và số dư

Chia từng biến số tiền và số dư theo tối đa 32 khoảng phân vị. Tính tỷ lệ gian lận của khoảng và làm trơn về tỷ lệ chung với hệ số 50. Tại từng giá trị chính xác đến cent, làm trơn tỷ lệ về tỷ lệ của khoảng với hệ số 20.

Giá trị chưa gặp dùng tỷ lệ của khoảng. Giá trị hiếm được kéo về tỷ lệ của khoảng mạnh hơn. Biên phân vị và các thống kê nhãn chỉ dùng phần fit tương ứng. Mỗi dòng train nhận mã hóa từ bốn fold mã hóa còn lại.

## Huấn luyện và gộp dự đoán

1. Trong mỗi fold, nhánh XGBoost chọn số cây trên phần giữ lại bên trong train, rồi huấn luyện lại trên toàn bộ phần train của fold.
2. Tái dựng phép chuẩn hóa và trọng số logistic gốc bằng 25 tệp dự đoán ngoài fold bên trong của mỗi nhánh gốc. Đối chiếu kết quả tái dựng với `FINAL` đã lưu.
3. Đưa nhánh XGBoost mới vào cùng `GLM-NEWTON`, áp đúng trọng số gốc của từng fold và chấm trên fold ngoài.
4. Tính thêm phương án trung bình xác suất 50/50 giữa `FINAL` gốc và bản thay nhánh sau khi đối chiếu ID và fold.

In [1]:
from pathlib import Path
import json

import pandas as pd
from IPython.display import display

VARIANT = 'RARE-TE'
MODE = "view"  # "view", "smoke", "full"
VIEW_RESULT = "full"  # "smoke" hoặc "full" khi MODE là view

if MODE not in {"view", "smoke", "full"} or VIEW_RESULT not in {"smoke", "full"}:
    raise ValueError("Chọn MODE là view, smoke hoặc full và VIEW_RESULT là smoke hoặc full.")

cwd = Path.cwd().resolve()
project = next(
    (path for path in (cwd, *cwd.parents)
     if (path / "train.csv").is_file()
     and (path / "s6e11_gpu" / "final_feature_optimization").is_dir()),
    None,
)
if project is None:
    raise FileNotFoundError("Mở notebook từ thư mục dự án hoặc một thư mục con.")

experiment_root = project / "s6e11_gpu" / "final_feature_optimization"
reference_root = project / "s6e11_gpu" / "plan_runs" / "results" / "full"
result_mode = VIEW_RESULT if MODE == "view" else MODE
run_root = experiment_root / "results" / result_mode

## Chạy thí nghiệm

Chế độ mặc định `view` chỉ đọc kết quả đã lưu. Chọn `smoke` để thử nhánh XGBoost trên mẫu nhỏ. Chọn `full` để chạy đủ 5 fold và đánh giá tổ hợp. Bảng 50/50 xuất hiện sau khi chạy `report_results.py` với đủ kết quả ba cấu hình.

In [2]:
import os
import subprocess
import sys

if MODE != "view":
    command = [sys.executable, "-u", str(experiment_root / "run_optimization.py"),
               "--variants", VARIANT]
    if MODE == "smoke":
        command.append("--smoke")
    environment = dict(os.environ, PYTHONIOENCODING="utf-8")
    subprocess.run(command, cwd=project, env=environment, check=True)
    if MODE == "full":
        subprocess.run(
            [sys.executable, "-u", str(experiment_root / "evaluate_final.py"),
             "--variants", VARIANT],
            cwd=project, env=environment, check=True,
        )

## Đọc kết quả

So nhánh XGBoost mới với `XGB-D03` gốc. So hai cách gộp với `FINAL` gốc. OOF là dự đoán ngoài fold. AP (Average Precision) tổng hợp độ chính xác và độ bao phủ theo xếp hạng.

In [3]:
def read_metrics(path):
    return json.loads(path.read_text(encoding="utf-8")) if path.is_file() else None


tree = read_metrics(run_root / f"XGB-D03-{VARIANT}" / "metrics.json")
blend = read_metrics(run_root / f"FINAL-{VARIANT}" / "metrics.json")
reference_tree = read_metrics(reference_root / "XGB-D03" / "metrics.json") if result_mode == "full" else None
reference_final = read_metrics(reference_root / "final" / "metrics.json") if result_mode == "full" else None
comparison = read_metrics(experiment_root / "results" / "comparison.json") if result_mode == "full" else None
anchor = next(
    (row for row in comparison["anchored"] if row["name"] == f"50/50 FINAL + {VARIANT}"),
    None,
) if comparison else None
if tree is None:
    print(f"Chưa có kết quả XGB-D03-{VARIANT} ở chế độ {result_mode}.")
if result_mode == "full" and blend is None:
    print(f"Chưa có kết quả FINAL-{VARIANT}.")
if result_mode == "full" and anchor is None:
    print("Bảng 50/50 sẽ xuất hiện khi báo cáo tổng hợp sẵn sàng.")

comparisons = [(reference_tree, None), (tree, reference_tree),
               (reference_final, None), (blend, reference_final), (anchor, reference_final)]
rows, fold_rows = [], {}
for candidate, baseline in comparisons:
    if candidate is None:
        continue
    if baseline and candidate is not anchor:
        for key in ("rows", "fold_hash", "early_stop_hash", "inputs"):
            if candidate[key] != baseline[key]:
                raise ValueError(f"Cấu hình {candidate['name']} khác đối chứng ở {key}.")
    delta_auc = candidate["oof_auc"] - baseline["oof_auc"] if baseline else None
    delta_ap = candidate["average_precision"] - baseline["average_precision"] if baseline else None
    rows.append({
        "Cấu hình": candidate["name"],
        "OOF ROC AUC": candidate["oof_auc"],
        "Δ AUC": delta_auc,
        "AP": candidate["average_precision"],
        "Δ AP": delta_ap,
        "Log-loss": candidate["log_loss"],
    })
    per_fold = {entry["fold"]: entry["auc"] for entry in candidate["folds"]}
    if baseline:
        baseline_scores = {entry["fold"]: entry["auc"] for entry in baseline["folds"]}
        if per_fold.keys() != baseline_scores.keys():
            raise ValueError(f"Fold của {candidate['name']} không khớp đối chứng.")
        fold_rows[candidate["name"]] = {
            fold: value - baseline_scores[fold] for fold, value in per_fold.items()
        }

if rows:
    display(pd.DataFrame(rows).set_index("Cấu hình").round(6))
if fold_rows:
    delta_frame = pd.DataFrame(fold_rows).rename_axis("Fold").sort_index()
    display(delta_frame.round(6))
    display((delta_frame > 0).sum().to_frame("Số fold tăng"))
if tree:
    devices = sorted({entry["device"] for entry in tree["folds"]})
    print("Thiết bị nhánh XGBoost", ", ".join(devices))
    print("Số cây theo fold", ", ".join(str(entry["rounds"]) for entry in tree["folds"]))

Thiết bị nhánh XGBoost cuda:0: NVIDIA GeForce RTX 4060 Laptop GPU
Số cây theo fold 2251, 1395, 1751, 2258, 2335


,OOF ROC AUC,Δ AUC,AP,Δ AP,Log-loss
Cấu hình,,,,,
XGB-D03,0.950182,NaN,0.528763,NaN,0.044422
XGB-D03-RARE-TE,0.950063,-0.000120,0.528477,-0.000285,0.044447
FINAL,0.950345,NaN,0.529376,NaN,0.044347
FINAL-RARE-TE,0.950260,-0.000086,0.529168,-0.000208,0.044364
50/50 FINAL + RARE-TE,0.950311,-0.000034,0.529320,-0.000056,0.044352


,XGB-D03-RARE-TE,FINAL-RARE-TE,50/50 FINAL + RARE-TE
Fold,,,
1,-0.000245,-0.000189,-0.000085
2,0.000023,0.000009,0.000018
3,-0.000050,-0.000036,-0.000013
4,-0.000020,-0.000018,-0.000000
5,-0.000167,-0.000108,-0.000043


,Số fold tăng
XGB-D03-RARE-TE,1
FINAL-RARE-TE,1
50/50 FINAL + RARE-TE,1


## Tiêu chí chọn ứng viên

Mốc sàng lọc theo kế hoạch là **Δ OOF ROC AUC ≥ +0.000100** và **tăng ở ít nhất 4/5 fold**. Các phép so dùng lại 5 fold phát triển của dự án và trọng số gộp cố định.

[Báo cáo tổng hợp ba nhóm](../results/ket_qua_toi_uu_dac_trung_final.md)